# Find Concurrent Writes With Vector Clocks

Companion notebook for the [Vector Clocks and Causality](https://codemia.io/courses/distributed_systems/dst_vector_clocks_and_causality) lesson on Codemia.

**What you will build**: a small message passing simulator that stamps every event with a Lamport clock and a vector clock. You will check which pairs of events are causally ordered and which are concurrent, see exactly what a Lamport clock cannot tell you, draw the space time diagram, and then use vector clocks the way a leaderless store does: to keep concurrent writes as siblings instead of silently dropping one with last writer wins.

No GPU. Standard library, `numpy` and `matplotlib` only.

## Setup

In [ ]:
%pip install -q numpy matplotlib

In [ ]:
import random
from itertools import combinations

import numpy as np
import matplotlib.pyplot as plt

random.seed(7)
np.random.seed(7)

## 1. A message passing simulator

Each of `n` processes performs a sequence of steps. A step is a local event, a send to a random peer, or a receive of a message already in flight. Every event gets two stamps:

- a **Lamport clock**: increment before each event, and on receive take `max(local, message) + 1`;
- a **vector clock**: increment your own entry before each event, and on receive take the elementwise maximum with the message vector first.

We also record the true causal history of every event, built from the same rules, so we can check the clocks against ground truth.

In [ ]:
def simulate(n=3, steps=30, p_send=0.35, p_recv=0.35, seed=1):
    rng = random.Random(seed)
    lamport = [0] * n
    vector = [[0] * n for _ in range(n)]
    history = [set() for _ in range(n)]   # events that happen before the next event on each process
    in_flight = []                         # (dst, lamport, vector, history, send_event_id)
    events = []                            # dicts with id, proc, kind, L, V, hist
    for _ in range(steps):
        p = rng.randrange(n)
        mine = [m for m in in_flight if m[0] == p]
        r = rng.random()
        if mine and r < p_recv:
            msg = mine[0]
            in_flight.remove(msg)
            _, mL, mV, mH, src_id = msg
            lamport[p] = max(lamport[p], mL) + 1
            vector[p] = [max(a, b) for a, b in zip(vector[p], mV)]
            vector[p][p] += 1
            history[p] |= mH
            kind, link = "recv", src_id
        else:
            lamport[p] += 1
            vector[p][p] += 1
            kind, link = ("send", None) if r < p_recv + p_send else ("local", None)
        eid = len(events)
        events.append(dict(id=eid, proc=p, kind=kind, L=lamport[p], V=tuple(vector[p]), hist=frozenset(history[p]), link=link))
        history[p] = history[p] | {eid}
        if kind == "send":
            dst = rng.choice([q for q in range(n) if q != p])
            in_flight.append((dst, lamport[p], tuple(vector[p]), frozenset(history[p]), eid))
    return events

events = simulate()
for e in events[:12]:
    print(e["id"], "P%d" % e["proc"], e["kind"].ljust(5), "L=%-3d" % e["L"], "V=%s" % (e["V"],))

## 2. Comparing events

Vector clock `a` happens before `b` when every entry of `a` is at most the matching entry of `b` and at least one is strictly smaller. If neither happens before the other, the events are **concurrent**. Ground truth says `a` happens before `b` when `a` is in the causal history of `b`.

Check that the vector clock agrees with the truth on every pair.

In [ ]:
def vc_before(a, b):
    return all(x <= y for x, y in zip(a, b)) and a != b

def relation(e, f):
    if vc_before(e["V"], f["V"]):
        return "before"
    if vc_before(f["V"], e["V"]):
        return "after"
    return "concurrent"

mismatch = 0
for e, f in combinations(events, 2):
    truth = "before" if e["id"] in f["hist"] else "after" if f["id"] in e["hist"] else "concurrent"
    mismatch += relation(e, f) != truth
print("pairs checked:", len(events) * (len(events) - 1) // 2, " mismatches:", mismatch)

### What a Lamport clock cannot tell you

A Lamport clock guarantees that if `a` happens before `b` then `L(a) < L(b)`. The converse fails: a smaller Lamport timestamp does not mean the event came first causally. Count how often `L(a) < L(b)` holds for a pair that is actually concurrent.

In [ ]:
lam_less = conc_with_less = 0
for e, f in combinations(events, 2):
    if e["L"] == f["L"]:
        continue
    lo, hi = (e, f) if e["L"] < f["L"] else (f, e)
    lam_less += 1
    conc_with_less += relation(lo, hi) == "concurrent"
print("pairs with different Lamport stamps:", lam_less)
print("of which actually concurrent:      ", conc_with_less, "(%.0f%%)" % (100 * conc_with_less / lam_less))

## 3. The space time diagram

Each horizontal line is a process, each dot an event, each arrow a message. Pick one event and colour every other event by its relation to it: its past, its future, and the concurrent events that neither could have influenced nor been influenced by it.

In [ ]:
def draw(events, focus):
    n = max(e["proc"] for e in events) + 1
    x = {e["id"]: i for i, e in enumerate(events)}
    fig, ax = plt.subplots(figsize=(11, 3.2))
    for p in range(n):
        ax.axhline(p, color="#bbb", lw=1, zorder=0)
    for e in events:
        if e["kind"] == "recv":
            s = events[e["link"]]
            ax.annotate("", xy=(x[e["id"]], e["proc"]), xytext=(x[s["id"]], s["proc"]),
                        arrowprops=dict(arrowstyle="->", color="#999", lw=1))
    colours = {"before": "#2b6cb0", "after": "#2f855a", "concurrent": "#dd6b20"}
    f = events[focus]
    for e in events:
        c = "black" if e["id"] == focus else colours[relation(e, f)]
        ax.scatter(x[e["id"]], e["proc"], s=60 if e["id"] == focus else 30, color=c, zorder=3)
    ax.set_yticks(range(n), ["P%d" % p for p in range(n)])
    ax.set_xticks([])
    ax.set_title("Event %d (black): past in blue, future in green, concurrent in orange" % focus)
    plt.show()

draw(events, focus=len(events) // 2)

## 4. Siblings instead of lost writes

A leaderless store accepts writes at any replica. Two clients that read the same version and both write back have made **concurrent** writes. Last writer wins keeps the one with the larger wall clock timestamp and drops the other, even when the clocks are skewed. A store that stamps versions with vector clocks can see that neither write descends from the other and keep both as siblings for the application to merge.

Simulate clients doing read, modify, write cycles against random replicas, with replica clocks skewed by up to `skew_ms`, and count how many acknowledged writes each policy loses.

In [ ]:
def descends(a, b):
    da, db = dict(a), dict(b)
    return all(da.get(k, 0) >= v for k, v in db.items())

def run_store(clients=4, rounds=200, replicas=3, skew_ms=20, think_ms=5, seed=3):
    rng = random.Random(seed)
    skew = [rng.uniform(-skew_ms, skew_ms) for _ in range(replicas)]
    lww = (-1e18, None, ())              # (timestamp, write id, vector clock)
    siblings = {(): None}                # vector clock -> write id
    acked = lost_lww = 0
    now = 0.0
    for w in range(rounds):
        now += rng.expovariate(1 / think_ms)
        c = rng.randrange(clients)
        r = rng.randrange(replicas)
        # read: the client sees the current sibling set and merges their clocks
        ctx = {}
        for vc in siblings:
            for k, v in vc:
                ctx[k] = max(ctx.get(k, 0), v)
        # a slow client may have read an older version: drop the newest entry sometimes
        if rng.random() < 0.3 and ctx:
            k = rng.choice(list(ctx)); ctx[k] -= 1
        ctx[r] = ctx.get(r, 0) + 1
        new_vc = tuple(sorted(ctx.items()))
        # vector clock store: drop only siblings the new write descends from
        siblings = {vc: wid for vc, wid in siblings.items() if not descends(new_vc, vc)}
        siblings[new_vc] = w
        acked += 1
        # last writer wins on skewed wall clocks: a write is lost if it is rejected,
        # or if it overwrites a winner the writer never saw
        ts = now + skew[r]
        if ts > lww[0]:
            if lww[1] is not None and not descends(new_vc, lww[2]):
                lost_lww += 1
            lww = (ts, w, new_vc)
        else:
            lost_lww += 1
    return acked, lost_lww, len(siblings)

for skew_ms in [0, 5, 20, 50]:
    acked, lost, sib = run_store(skew_ms=skew_ms)
    print("skew %3d ms: acked %d, LWW silently dropped %3d, VC store holds %d sibling(s) at the end" % (skew_ms, acked, lost, sib))

Even with no skew, last writer wins drops a write whenever a concurrent write overwrites it, because the second writer never saw the first. With skew, it also drops writes that were not concurrent at all, because a replica with a fast clock outranks a later write elsewhere. The vector clock store never drops an acknowledged write without a successor that saw it.

Plot the LWW losses against skew.

In [ ]:
skews = [0, 2, 5, 10, 20, 35, 50, 80]
losses = [np.mean([run_store(skew_ms=s, seed=k)[1] for k in range(20)]) for s in skews]
plt.figure(figsize=(6, 3.2))
plt.plot(skews, losses, marker="o")
plt.xlabel("max clock skew between replicas (ms)")
plt.ylabel("writes dropped by LWW (of 200)")
plt.title("Last writer wins loses more as clocks drift")
plt.grid(alpha=0.3)
plt.show()

## 5. Exercises

1. Raise `n` to 6 processes in `simulate`. How does the share of concurrent pairs change, and why?
2. Vector clocks grow with the number of writers. Change `run_store` so that clients, not replicas, own vector entries, and measure the clock size after 1,000 writes from 50 clients.
3. Replace the sibling set with a merge function, such as union for a shopping cart, and confirm that no item added by any acknowledged write disappears.